# T45D / E19 — Định vị bằng cắt bỏ ngữ cảnh, so với chú ý

**Câu hỏi hội đồng dễ hỏi nhất:** *chú ý có phải lời giải thích không?* ContextCite (NeurIPS 2024)
cho rằng trọng số chú ý thường không đáng tin để quy trách nhiệm cho nguồn; cách đáng tin là
**cắt bỏ**: bỏ một đoạn ngữ cảnh ra rồi đo xem câu trả lời của mô hình thay đổi bao nhiêu.

E19 làm đúng phép đó (kiểu ARC-JSD) trên **300 mẫu "có vàng" của E08** (ViWikiFC, ngữ cảnh mười
câu): mỗi mẫu đọc một lần đủ ngữ cảnh rồi đọc lại với **từng đoạn bị cắt**, đo độ lệch
Jensen–Shannon của phân phối trên các token câu trả lời. Đoạn làm lệch nhiều nhất là đoạn được quy
trách nhiệm. So hit@1 của cách này với các đầu chú ý **trên cùng 300 mẫu** — phần chú ý đã có sẵn
trong shard E08 trên máy, không chạy lại ở đây.

**Dự đoán ghi trước khi chạy** (khóa trong `src/vihallulens/attribution.py`, có ca kiểm thử):
cắt bỏ đạt hit@1 cao hơn đầu chú ý mạnh nhất, nhưng mức hơn nhỏ hơn mức chênh chi phí
(khoảng 10,9 lượt đọc mỗi mẫu so với 1).

## Vì sao chạy bfloat16

Cắt bỏ cần **phân phối đầu ra** của mô hình. Qwen2.5-7B ở float16 tràn số ở lớp 27 và đưa NaN vào
tầng dự đoán token (đo ở T40), nên phải chạy bfloat16 — T4 không có bf16 gốc nên chậm hơn khoảng
bốn lần. Chú ý thì không cần tầng đó, nên E08 chạy được float16.

## Thời gian

3.274 lượt đọc (300 mẫu × trung bình 10,9). **Ước tính 1,5–2,5 giờ GPU**; ô 6 chạy thử 3 mẫu và in
tốc độ thật, xem số đó trước khi để ô 7 chạy hết. Mỗi mẫu ghi xuống ngay, nên phiên đứt giữa chừng
chạy lại đúng các ô là làm tiếp phần còn lại.

## Cần tải về sau khi chạy

`/kaggle/working/e19_attribution.jsonl` → đặt vào `results/e19_attribution.jsonl` trên máy, rồi
chạy `python scripts/run_attribution.py report`. Tải cả notebook đã chạy làm minh chứng.

In [1]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

đã clone mới
/kaggle/working/vihallulens
commit: 17a2516 Merge pull request #1 from wsunicorn/lich-su-pr


In [2]:
# Ô 2 — cài đặt, ghim phiên bản theo uv.lock (requirements-kaggle.txt, từ 22/10/2026).
# `--no-deps` cố ý không kéo phụ thuộc nào về, nên rank-bm25 (BM25 dựng ngữ cảnh) nằm trong file ghim.
!pip install -q --no-deps -e .
!pip install -q -r requirements-kaggle.txt

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vihallulens (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vihallulens 0.1.0 requires pyvi, which is not installed.


In [3]:
# Ô 3 — TIỀN KIỂM. Vài giây, chạy trước mọi thứ. Chặn những gì phiên này không tự tạo được.
import importlib.util
import json
import sys
from pathlib import Path

import torch

sys.path.insert(0, "src")
from vihallulens.data.paths import find_raw_dir

problems = []
packages = ("rank_bm25", "torch", "transformers", "bitsandbytes", "pandas")
absent = [name for name in packages if importlib.util.find_spec(name) is None]
print(f"  gói phải có sẵn   : {'THIẾU ' + str(absent) if absent else 'đủ'}")
if absent:
    problems.append(f"thiếu gói {absent}")
print(f"  GPU               : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ'}")
if not torch.cuda.is_available():
    problems.append("chưa bật GPU T4")
try:
    raw = find_raw_dir()
    files = sorted(p.name for p in Path(raw).glob("viwikifc*"))
    print(f"  dữ liệu thô       : {raw} — {files or 'KHÔNG CÓ viwikifc'}")
    if not files:
        problems.append("không thấy file viwikifc trong dữ liệu thô")
except Exception as error:
    print(f"  dữ liệu thô       : KHÔNG TÌM THẤY ({error})")
    problems.append("chưa gắn dataset dữ liệu thô")
selection = json.loads(Path("configs/e19_attribution_samples.json").read_text(encoding="utf-8"))
print(f"  mẫu đã chọn       : {len(selection['samples'])} (hạt giống {selection['seed']})")
if problems:
    raise SystemExit("TIỀN KIỂM HỎNG: " + "; ".join(problems))
print("Tiền kiểm đạt.")

  gói phải có sẵn   : đủ
  GPU               : Tesla T4
  dữ liệu thô       : /kaggle/input/datasets/unicorn1209/vihallulens — ['viwikifc_dev.csv', 'viwikifc_test.csv', 'viwikifc_train.csv']
  mẫu đã chọn       : 300 (hạt giống 42)
Tiền kiểm đạt.


In [4]:
# Ô 4 — chuẩn bị dữ liệu và kiểm môi trường. Khoảng 2 phút, CPU.
!python scripts/probe_env.py
!python scripts/normalize_data.py --dataset viwikifc
!python scripts/split_data.py --only viwikifc
!python -m pytest tests/test_attribution.py tests/test_paired_contexts.py tests/test_revisions.py -q


MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : 17a2516 Merge pull request #1 from wsunicorn/lich-su-pr
  python           : 3.13.15
  torch            : 2.11.0+cu128
  transformers     : 5.17.0
  bitsandbytes     : 0.50.2
  accelerate       : 1.15.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv

CHUẨN HÓA VIWIKIFC
  nguồn                 : /kaggle/input/datasets/unicorn1209/vihallulens
  số dòng               : 20,919
  ngữ cảnh duy nhất     :

In [5]:
# Ô 5 — dựng kho truy xuất và ngữ cảnh mười câu như E08. Khoảng 2 phút, CPU.
# Lệnh run ở ô sau so mã băm từng ngữ cảnh với bản E08 đã trích và DỪNG nếu có ngữ cảnh khác.
!python scripts/build_evidence_corpus.py
!python scripts/build_retrieval_contexts.py --split dev


T16 — KHO TRUY XUẤT VIWIKIFC
  nguồn                 : /kaggle/input/datasets/unicorn1209/vihallulens
  câu bằng chứng        : 3,814
  bài Wikipedia         : 73
  độ dài câu (từ)       : trung vị 31, trung bình 35.2, dài nhất 251
  claim mỗi câu         : trung bình 5.5, tối đa 42
  đã ghi                : data/interim/viwikifc_evidence_corpus.parquet
  dựng chỉ mục BM25     : 0.10 s

--------------------------------------------------------------------------------
TRUY VẤN THỬ
--------------------------------------------------------------------------------

  claim : Thác nước Jog nằm trên sông Hằng.
  nhãn  : intrinsic  ·  bài: châu Á
    1. [ 25.4] Thác nước có chiều cao nước rơi cách mặt sông lớn nhất châu Á là thác nước Jog trê ← BẰNG CHỨNG VÀNG
    2. [ 14.9] Trên tuyến sông, vào mùa nước trung thì tàu thuyền có thể khai thác thuận lợi, san
    3. [ 13.7] Hệ thống sông Vu Gia - Thu Bồn với phần lớn lưu vực nằm trong địa giới tỉnh được đ
    4. [ 13.3] Wonsan là thành phố duyên 

In [6]:
# Ô 6 — chạy thử 3 mẫu để đo tốc độ thật. Khoảng 3 phút (gồm nạp mô hình).
# Ba mẫu này được ghi lại và ô 7 không chạy lại chúng.
!python scripts/run_attribution.py run --limit 3 --out /kaggle/working/e19_attribution.jsonl


E19 — ĐỊNH VỊ BẰNG CẮT BỎ NGỮ CẢNH (ARC-JSD)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct, NF4, bfloat16
  mẫu                   : 3 (đã có 0, còn 3)
  lượt đọc dự kiến      : 33 (1 + số đoạn mỗi mẫu)
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 3.01MB/s]
tokenizer_config.json: 100%|███████████████| 7.30k/7.30k [00:00<00:00, 20.0MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 49.2MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 114MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 150MB/s]
model.safetensors.index.json: 100%|████████| 27.8k/27.8k [00:00<00:00, 83.5MB/s]
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 4 files:   0%|                                   | 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|        |  0.00B / 7.42GB            
Reconstructing (incomplete total...):   0%|        |  0.00B / 

In [7]:
# Ô 7 — chạy đủ 300 mẫu. Ước tính 1,5–2,5 giờ; dòng tiến độ in tốc độ và thời gian còn lại.
!python scripts/run_attribution.py run --out /kaggle/working/e19_attribution.jsonl


E19 — ĐỊNH VỊ BẰNG CẮT BỎ NGỮ CẢNH (ARC-JSD)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct, NF4, bfloat16
  mẫu                   : 300 (đã có 3, còn 297)
  lượt đọc dự kiến      : 3,241 (1 + số đoạn mỗi mẫu)
Loading weights: 100%|████████████████████████| 339/339 [00:04<00:00, 76.32it/s]

  xong: 297 mẫu, lỗi 0, 165.5 phút -> /kaggle/working/e19_attribution.jsonl


In [8]:
# Ô 8 — kiểm file kết quả trước khi tải về.
import json
from pathlib import Path

path = Path("/kaggle/working/e19_attribution.jsonl")
rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"{len(rows)} mẫu, {sum(r['n_forward'] for r in rows):,} lượt đọc, "
      f"{sum(r['elapsed_ms'] for r in rows) / 60000:.1f} phút, "
      f"VRAM đỉnh {max(r['peak_vram_mb'] for r in rows):,.0f} MB")
print("Tải về: /kaggle/working/e19_attribution.jsonl -> results/e19_attribution.jsonl")

300 mẫu, 3,274 lượt đọc, 167.0 phút, VRAM đỉnh 5,659 MB
Tải về: /kaggle/working/e19_attribution.jsonl -> results/e19_attribution.jsonl
